# US Beta / Required Return — DB Price Source V4

**저장 테이블** : `us_required_return_result`  
**저장 형태**   : `date · ticker · indicator · value` (long-format)  
**저장 방법**   : `INSERT ... ON DUPLICATE KEY UPDATE` (upsert)  
**PRIMARY KEY** : `(date, ticker, indicator)`  
**주가 소스**   : DB 테이블 `us_stock_daily_market_cap` (indicator=`close_price`)

---

| 셀 | 단계 |
|----|------|
| 1  | 환경 설정 & 경로 자동 감지 (노트북/데스크탑 자동 판별) |
| 2  | 모듈 Import & 설정 상수 |
| 3  | DB 연결 함수 / 조회 함수 |
| 4  | DB에서 주가 fetch 함수 (us_stock_daily_market_cap) |
| 5  | RF(국채금리) fetch 함수 |
| 6  | Beta · Required Return 계산 함수 (merge_asof RF 조인) |
| 7  | MySQL 저장 함수 (sanitize + upsert) |
| 8  | 티커 1개 업데이트 함수 (update_one_ticker) |
| 9  | SPY / RF 공통 데이터 준비 |
| 10 | 배치 실행 (구간 지정 · 체크포인트 · 실패 재시도) |
| 11 | DB 조회 유틸 (pivot 조회) |
| 12 | 결과 조회 테스트 |

---
### V3 → V4 변경 사항
- **주가 소스 변경**: FMP API → DB `us_stock_daily_market_cap` (indicator=`close_price`)
- **SPY 시장 지수**: DB에서 조회, 없으면 FDR fallback
- **데이터 품질 검증**: 시계열 연속성 검사, 최소 거래일 수 체크, 이상값 필터
- **RF 조인 수정**: `merge_asof(direction=backward)` → 날짜 타입 불일치 & NaN 문제 해결
- **중복 방지**: (ticker, date) 기준 신규 데이터만 upsert
- **진행률**: `[idx/total] (pct%)` 실시간 출력

## Cell 1 · 환경 설정 & 경로 자동 감지

In [1]:
import sys, os, gc, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── 후보 프로젝트 루트 (노트북 / 데스크탑) ───────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",          # 노트북
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",   # 데스크탑
]

def _setup_path() -> str:
    """
    DATA/ 폴더를 포함하는 프로젝트 루트를 탐색해 sys.path 에 추가합니다.
    탐색 순서:
      1) cwd / __file__ 상위 경로 중 DATA/ 를 포함하는 첫 번째 경로
      2) _CANDIDATE_ROOTS 에서 DATA/ 가 있는 첫 번째 경로
    """
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()

    # 현재 디렉토리 및 상위 경로 탐색
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root

    # 하드코딩 후보 경로 탐색
    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root 후보 경로 : {candidate}")
            return candidate

    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다.\n"
        "_CANDIDATE_ROOTS 를 현재 환경에 맞게 수정하거나 "
        "노트북을 프로젝트 루트 아래에서 실행하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")
print(f"[확인] DATA 경로    : {os.path.join(_ROOT, 'DATA')}")

[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] DATA 경로    : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


## Cell 2 · 모듈 Import & 설정 상수

In [2]:
# ── 표준 라이브러리 ───────────────────────────────────────────────
import math
import time
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple
from pandas.tseries.offsets import BDay

# ── 외부 라이브러리 ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import pymysql
import FinanceDataReader as fdr
from IPython.display import display

# ── 내부 모듈 (경로는 Cell 1에서 자동 감지된 _ROOT 기준) ──────────
from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list as DEFAULT_TICKER_LIST

# ── 로그 유틸 ─────────────────────────────────────────────────────
def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}")

# ══════════════════════════════════════════════════════════════════
#  설정 상수 — 여기만 수정하세요
# ══════════════════════════════════════════════════════════════════
DB_NAME       = "investar"
TABLE_RESULT  = "us_required_return_result"       # 결과 저장 테이블
TABLE_PRICE   = "us_stock_daily_market_cap"        # 주가 소스 테이블
PRICE_IND     = "close_price"                      # 주가 indicator 명
DEFAULT_PORT  = 3307
MARKET_TICKER = "SPY"                             # 시장 지수 티커

MIN_START_DATE = "2015-01-01"                     # 계산 시작일

# Beta 윈도우 (영업일 기준)
BETA_WINDOWS         = [252, 750, 1250]           # 1y / 3y / 5y
MAX_ROLLING_WINDOW   = max(BETA_WINDOWS)          # 1250
ROLLING_WARMUP_BDAYS = MAX_ROLLING_WINDOW + 80    # 계산 워밍업 기간

# 데이터 품질 기준
MIN_PRICE_ROWS  = 300    # 최소 거래일 수 (미달 시 제외)
MAX_PRICE_GAP   = 30     # 최대 허용 연속 공백 거래일

# 저장 모드: "minimal" (Re + beta) / "full" (모든 중간값 포함)
STORE_MODE = "minimal"

# 체크포인트
CHECKPOINT_DIR    = "_batch_checkpoint_v4"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DEFAULT_DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")
DEFAULT_FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")

# ══════════════════════════════════════════════════════════════════
print(f"[OK] Import 완료")
print(f"[OK] DEFAULT_TICKER_LIST: {len(DEFAULT_TICKER_LIST):,}개")
print(f"[설정] TABLE_RESULT={TABLE_RESULT}")
print(f"[설정] TABLE_PRICE={TABLE_PRICE}  PRICE_IND={PRICE_IND}")
print(f"[설정] STORE_MODE={STORE_MODE}  BETA_WINDOWS={BETA_WINDOWS}")
print(f"[설정] MIN_START_DATE={MIN_START_DATE}")

[OK] Import 완료
[OK] DEFAULT_TICKER_LIST: 2,000개
[설정] TABLE_RESULT=us_required_return_result
[설정] TABLE_PRICE=us_stock_daily_market_cap  PRICE_IND=close_price
[설정] STORE_MODE=minimal  BETA_WINDOWS=[252, 750, 1250]
[설정] MIN_START_DATE=2015-01-01


## Cell 3 · DB 연결 함수 / 조회 함수

In [3]:
# ── DB 접속 정보 ──────────────────────────────────────────────────
db_info = get_db_info()


def get_conn(db_info: Dict[str, Any]):
    """pymysql 연결 생성 (DictCursor)"""
    return pymysql.connect(
        host        = db_info["host"],
        port        = db_info.get("port", DEFAULT_PORT),
        user        = db_info["user"],
        password    = db_info["password"],
        db          = db_info.get("database", DB_NAME),
        charset     = "utf8mb4",
        autocommit  = False,
        cursorclass = pymysql.cursors.DictCursor,
    )


def get_minmax_date_in_result(
    db_info: Dict[str, Any],
    ticker: str,
    indicator: str,
) -> Tuple[Optional[pd.Timestamp], Optional[pd.Timestamp]]:
    """결과 테이블에서 (ticker, indicator) 의 MIN/MAX date 조회"""
    sql = f"""
        SELECT MIN(date) AS first_date, MAX(date) AS last_date
        FROM   {TABLE_RESULT}
        WHERE  ticker=%s AND indicator=%s;
    """
    conn = get_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (ticker, indicator))
            row = cur.fetchone()
            first_dt = row["first_date"] if row else None
            last_dt  = row["last_date"]  if row else None
    finally:
        conn.close()

    first_dt = pd.to_datetime(first_dt) if first_dt is not None else None
    last_dt  = pd.to_datetime(last_dt)  if last_dt  is not None else None
    return first_dt, last_dt


# ── 연결 테스트 ───────────────────────────────────────────────────
try:
    _c = get_conn(db_info)
    with _c.cursor() as _cur:
        _cur.execute("SELECT 1")
    _c.close()
    log("DB", f"연결 성공  host={db_info.get('host')}  port={db_info.get('port')}  db={db_info.get('database')}")
except Exception as _e:
    log("DB", f"연결 실패: {_e}")

[16:07:31][DB] 연결 성공  host=192.168.0.230  port=3307  db=investar


## Cell 4 · DB에서 주가 fetch 함수

- **소스**: `us_stock_daily_market_cap` 테이블, `indicator = 'close_price'`
- **데이터 품질 검증**: 최소 거래일 수, 최대 연속 공백, 이상 수익률 필터
- **SPY fallback**: DB에 SPY 없으면 FDR에서 직접 fetch

In [4]:
def fetch_price_from_db(
    db_info: Dict[str, Any],
    ticker: str,
    start_date: Optional[str] = None,
    end_date: Optional[str]   = None,
) -> pd.DataFrame:
    """
    us_stock_daily_market_cap 테이블에서 close_price 시계열 조회.

    Returns
    -------
    DataFrame  columns: [date, price]   (정렬됨)
    빈 DataFrame이면 데이터 없음 또는 품질 불량
    """
    where  = ["ticker = %s", "indicator = %s"]
    params = [ticker, PRICE_IND]

    if start_date:
        where.append("date >= %s")
        params.append(start_date)
    if end_date:
        where.append("date <= %s")
        params.append(end_date)

    sql = f"""
        SELECT date, value AS price
        FROM   {TABLE_PRICE}
        WHERE  {' AND '.join(where)}
        ORDER  BY date;
    """
    conn = get_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            rows = cur.fetchall()
            df = pd.DataFrame(rows)
    finally:
        conn.close()

    if df.empty:
        return pd.DataFrame(columns=["date", "price"])

    df["date"]  = pd.to_datetime(df["date"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = (
        df.dropna(subset=["date", "price"])
        .drop_duplicates("date")
        .sort_values("date")
        .reset_index(drop=True)
    )
    return df


def validate_price_series(
    df: pd.DataFrame,
    ticker: str,
    min_rows: int  = MIN_PRICE_ROWS,
    max_gap: int   = MAX_PRICE_GAP,
) -> Tuple[bool, str]:
    """
    주가 시계열 품질 검증.

    Returns
    -------
    (True, "OK") or (False, 실패 사유)
    """
    if df is None or df.empty:
        return False, f"{ticker}: 데이터 없음"

    # 1) 최소 거래일 수
    if len(df) < min_rows:
        return False, f"{ticker}: 거래일 수 부족 ({len(df)} < {min_rows})"

    # 2) 음수/0 가격 제거 후 재확인
    df = df[df["price"] > 0]
    if len(df) < min_rows:
        return False, f"{ticker}: 유효 가격 부족 (양수 가격 {len(df)}개)"

    # 3) 최대 연속 공백 (영업일 기준 gap)
    dates = pd.to_datetime(df["date"]).sort_values()
    gaps  = dates.diff().dt.days.dropna()
    max_observed_gap = gaps.max()
    if max_observed_gap > max_gap:
        return False, f"{ticker}: 최대 날짜 공백 {int(max_observed_gap)}일 (기준 {max_gap}일)"

    return True, "OK"


def fetch_fdr_price_fallback(
    symbol: str,
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """FDR fallback (SPY 등 DB에 없을 때 사용). 반환: [date, price]"""
    try:
        df = fdr.DataReader(symbol, start_date, end_date)
    except Exception:
        return pd.DataFrame(columns=["date", "price"])

    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "price"])

    df = df.copy().reset_index()
    date_col = next((c for c in df.columns if c.lower() in ("date", "index")), df.columns[0])
    df = df.rename(columns={date_col: "date"})

    for pc in ("Adj Close", "adj close", "Close", "close"):
        if pc in df.columns:
            price_col = pc
            break
    else:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if not num_cols:
            return pd.DataFrame(columns=["date", "price"])
        price_col = num_cols[0]

    out = df[["date", price_col]].rename(columns={price_col: "price"})
    out["date"]  = pd.to_datetime(out["date"], errors="coerce")
    out["price"] = pd.to_numeric(out["price"], errors="coerce")
    out = out.dropna(subset=["date", "price"]).drop_duplicates("date").sort_values("date")
    return out


print("[OK] fetch_price_from_db / validate_price_series / fetch_fdr_price_fallback 정의 완료")

# ── 간단 테스트 ───────────────────────────────────────────────────
_test = fetch_price_from_db(db_info, "AAPL", start_date="2024-01-01")
if not _test.empty:
    log("TEST", f"AAPL DB 주가: {len(_test)}행  ({_test['date'].iloc[0].date()} ~ {_test['date'].iloc[-1].date()})")
else:
    log("TEST", "[WARN] AAPL DB 주가 없음 — TABLE_PRICE / PRICE_IND 설정 확인 필요")

[OK] fetch_price_from_db / validate_price_series / fetch_fdr_price_fallback 정의 완료
[16:07:31][TEST] AAPL DB 주가: 566행  (2024-01-02 ~ 2026-04-06)


## Cell 5 · RF(국채금리) fetch 함수

In [5]:
def fetch_us_treasury_yields(
    start_date: str,
    end_date: Optional[str] = None,
) -> pd.DataFrame:
    """
    1y / 3y / 5y 미국 국채 수익률을 FRED 또는 Yahoo 로 fetch.
    반환: index=date(Timestamp), columns=[rf_1y, rf_3y, rf_5y]  (소수 단위, 예: 0.045)
    """
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    # 1순위: FRED
    try:
        y1 = fdr.DataReader("FRED:DGS1", start_date, end_date)
        y3 = fdr.DataReader("FRED:DGS3", start_date, end_date)
        y5 = fdr.DataReader("FRED:DGS5", start_date, end_date)
        idx = y1.index.union(y3.index).union(y5.index)
        out = pd.DataFrame(index=idx).sort_index()
        out["rf_1y"] = pd.to_numeric(y1.iloc[:, 0], errors="coerce") / 100.0
        out["rf_3y"] = pd.to_numeric(y3.iloc[:, 0], errors="coerce") / 100.0
        out["rf_5y"] = pd.to_numeric(y5.iloc[:, 0], errors="coerce") / 100.0
        log("RF", f"FRED 로드 성공: {len(out)}행")
        return out
    except Exception:
        pass

    # 2순위: Yahoo (^IRX=1y, ^FVX=5y)
    try:
        y1 = fdr.DataReader("^IRX", start_date, end_date)
        y5 = fdr.DataReader("^FVX", start_date, end_date)
        idx = y1.index.union(y5.index)
        out = pd.DataFrame(index=idx).sort_index()
        out["rf_1y"] = pd.to_numeric(y1["Close"], errors="coerce") / 100.0
        out["rf_5y"] = pd.to_numeric(y5["Close"], errors="coerce") / 100.0
        # 3y = 1y ~ 5y 선형 보간
        out["rf_3y"] = out["rf_1y"] + (out["rf_5y"] - out["rf_1y"]) * (3 - 1) / (5 - 1)
        log("RF", f"Yahoo fallback 로드 성공: {len(out)}행")
        return out
    except Exception as e:
        log("RF", f"[WARN] 국채 금리 로드 실패: {e}")
        return pd.DataFrame(columns=["rf_1y", "rf_3y", "rf_5y"])


print("[OK] fetch_us_treasury_yields 함수 정의 완료")

[OK] fetch_us_treasury_yields 함수 정의 완료


## Cell 6 · Beta · Required Return 계산 함수

- RF 조인: `merge_asof(direction=backward)` → 날짜 타입 불일치 & 공백 NaN 문제 완전 해결
- CAPM: `Re = Rf + β × (E[Rm] - Rf)` (1y/3y/5y 기준)

In [6]:
def rolling_beta(
    ret_stock: pd.Series,
    ret_mkt: pd.Series,
    window: int,
) -> pd.Series:
    """Rolling Beta = Cov(ret_stock, ret_mkt) / Var(ret_mkt)"""
    cov = ret_stock.rolling(window).cov(ret_mkt)
    var = ret_mkt.rolling(window).var()
    return cov / var


def build_features(
    price_stock_df: pd.DataFrame,   # columns: [date, price_stock]
    price_mkt_df: pd.DataFrame,     # columns: [date, price_mkt]
    rf_df: pd.DataFrame,            # index=date(Timestamp), columns=[rf_1y, rf_3y, rf_5y]
) -> pd.DataFrame:
    """
    Beta 및 Required Return (CAPM) 계산.

    저장 지표:
      minimal : beta_252 / beta_750 / beta_1250 / Re_1y / Re_3y / Re_5y
      full    : 위 + price_mkt / ret_stock / ret_mkt / rf_* / E_Rm_*
    """
    df = pd.merge(price_stock_df, price_mkt_df, on="date", how="inner")
    df = df.sort_values("date").drop_duplicates("date").reset_index(drop=True)

    df["price_stock"] = pd.to_numeric(df["price_stock"], errors="coerce")
    df["price_mkt"]   = pd.to_numeric(df["price_mkt"],   errors="coerce")
    df = df.dropna(subset=["price_stock", "price_mkt"])

    if df.empty:
        return pd.DataFrame()

    df["ret_stock"] = df["price_stock"].pct_change()
    df["ret_mkt"]   = df["price_mkt"].pct_change()

    # Rolling Beta
    for w in BETA_WINDOWS:
        df[f"beta_{w}"] = rolling_beta(df["ret_stock"], df["ret_mkt"], w)

    # ── RF 조인 (merge_asof 방식 — 날짜 타입 불일치 & 공백 완전 해결) ──
    if rf_df is None or rf_df.empty:
        df["rf_1y"] = np.nan
        df["rf_3y"] = np.nan
        df["rf_5y"] = np.nan
    else:
        rf2 = rf_df.copy().reset_index()
        rf2.columns = ["date", "rf_1y", "rf_3y", "rf_5y"]
        # ★ timezone 제거 + normalize → date 타입 통일
        rf2["date"] = pd.to_datetime(rf2["date"]).dt.normalize().dt.tz_localize(None)
        rf2 = rf2.dropna(subset=["rf_1y", "rf_3y", "rf_5y"], how="all").sort_values("date")

        df["date"] = pd.to_datetime(df["date"]).dt.normalize()

        # RF 없는 날짜 → 가장 가까운 과거값으로 자동 채움
        df = pd.merge_asof(
            df.sort_values("date"),
            rf2[["date", "rf_1y", "rf_3y", "rf_5y"]],
            on="date",
            direction="backward",
        )

    # 기대 시장 수익률 (연환산)
    df["E_Rm_1y"] = df["ret_mkt"].rolling(252).mean()  * 252
    df["E_Rm_3y"] = df["ret_mkt"].rolling(750).mean()  * 252
    df["E_Rm_5y"] = df["ret_mkt"].rolling(1250).mean() * 252

    # CAPM Required Return: Re = Rf + β × (E[Rm] - Rf)
    df["Re_1y"] = df["rf_1y"] + df["beta_252"]  * (df["E_Rm_1y"] - df["rf_1y"])
    df["Re_3y"] = df["rf_3y"] + df["beta_750"]  * (df["E_Rm_3y"] - df["rf_3y"])
    df["Re_5y"] = df["rf_5y"] + df["beta_1250"] * (df["E_Rm_5y"] - df["rf_5y"])

    return df


print("[OK] rolling_beta / build_features 함수 정의 완료")
print(f"     BETA_WINDOWS = {BETA_WINDOWS}")
print(f"     저장 지표 (minimal): beta_252/750/1250, Re_1y/3y/5y")

[OK] rolling_beta / build_features 함수 정의 완료
     BETA_WINDOWS = [252, 750, 1250]
     저장 지표 (minimal): beta_252/750/1250, Re_1y/3y/5y


## Cell 7 · MySQL 저장 함수 (sanitize + upsert)

- NaN / inf → `None` 변환 후 저장
- `INSERT ... ON DUPLICATE KEY UPDATE value = VALUES(value)`
- PK `(date, ticker, indicator)` 기준 신규 데이터만 업데이트

In [7]:
def _to_mysql_float(x: Any) -> Optional[float]:
    """NaN / inf → None, 그 외 float 반환"""
    if x is None:
        return None
    try:
        v = float(x)
    except Exception:
        return None
    return None if (math.isnan(v) or math.isinf(v)) else v


def sanitize_long_for_mysql(df: pd.DataFrame) -> pd.DataFrame:
    """저장 전 데이터 정제: 날짜/타입 변환, NaN/중복 제거"""
    out = df.copy()
    out["date"]      = pd.to_datetime(out["date"], errors="coerce")
    out = out[out["date"].notna()]
    out["date"]      = out["date"].dt.date
    out["ticker"]    = out["ticker"].astype(str)
    out["indicator"] = out["indicator"].astype(str)
    out = out[~out["ticker"].str.lower().isin(["nan", "none"])]
    out = out[~out["indicator"].str.lower().isin(["nan", "none"])]
    out["value"]     = pd.to_numeric(out["value"], errors="coerce")
    out["value"]     = out["value"].apply(_to_mysql_float)
    out = out.drop_duplicates(subset=["date", "ticker", "indicator"])
    return out


def upsert_long_df(
    db_info: Dict[str, Any],
    long_df: pd.DataFrame,
    batch_size_rows: int   = 50_000,
    batch_size_ticker: int = 50,
    drop_null_values: bool = True,
) -> int:
    """
    long_df → DB upsert.
    PK (date, ticker, indicator) 중복 시 value 업데이트.
    Returns: 저장 행 수
    """
    if long_df is None or long_df.empty:
        return 0

    df = sanitize_long_for_mysql(long_df)
    if drop_null_values:
        df = df.dropna(subset=["value"])
    if df.empty:
        return 0

    tickers = sorted(df["ticker"].unique())
    total_saved = 0

    insert_sql = f"""
        INSERT INTO {TABLE_RESULT} (date, ticker, indicator, value)
        VALUES (%s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            value = VALUES(value);
    """

    for i in range(0, len(tickers), batch_size_ticker):
        batch_tickers = tickers[i : i + batch_size_ticker]
        batch = (
            df[df["ticker"].isin(batch_tickers)]
            .sort_values(["ticker", "date", "indicator"])
        )
        rows = list(batch[["date", "ticker", "indicator", "value"]].itertuples(index=False, name=None))

        conn = get_conn(db_info)
        try:
            with conn.cursor() as cur:
                for j in range(0, len(rows), batch_size_rows):
                    chunk = rows[j : j + batch_size_rows]
                    for _r in chunk[:10]:  # NaN/inf 샘플 검사
                        vv = _r[3]
                        if isinstance(vv, float) and (math.isnan(vv) or math.isinf(vv)):
                            raise ValueError(f"NaN/inf 발견: {_r}")
                    cur.executemany(insert_sql, chunk)
            conn.commit()
            log("DB", f"upsert batch {i//batch_size_ticker+1}: tickers={len(batch_tickers)}, rows={len(rows):,}")
            total_saved += len(rows)
        except Exception:
            conn.rollback()
            raise
        finally:
            conn.close()

    return total_saved


# ── 체크포인트 유틸 ───────────────────────────────────────────────
def _load_set(path: str) -> set:
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}

def _append_line(path: str, line: str) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(line.strip() + "\n")


print("[OK] sanitize_long_for_mysql / upsert_long_df 함수 정의 완료")

[OK] sanitize_long_for_mysql / upsert_long_df 함수 정의 완료


## Cell 8 · 티커 1개 업데이트 함수 (update_one_ticker)

- DB에서 주가 조회 → 품질 검증 → Beta/Re 계산 → 신규 구간만 저장
- `last_saved_date` 이후 데이터만 저장 (중복 저장 방지)

In [8]:
def update_one_ticker(
    db_info: Dict[str, Any],
    ticker: str,
    spy_price_df: pd.DataFrame,   # columns: [date, price_mkt]
    rf_df: pd.DataFrame,          # index=date(Timestamp), columns=[rf_1y, rf_3y, rf_5y]
    today_iso: Optional[str]  = None,
    min_start_date: str       = MIN_START_DATE,
    store_mode: str           = "minimal",
) -> Tuple[bool, str]:
    """
    티커 1개에 대해 DB 주가 조회 → 품질 검증 → 계산 → DB 저장 수행.

    Returns
    -------
    (True, 성공 메시지) or (False, 실패 메시지)
    """
    if today_iso is None:
        today_iso = datetime.utcnow().date().isoformat()

    if spy_price_df is None or spy_price_df.empty:
        return False, "SPY price df 없음"

    # 0) 마지막 저장일 확인 → 신규 구간만 저장
    _, last_saved = get_minmax_date_in_result(db_info, ticker, "Re_5y")

    if last_saved is not None:
        # 롤링 워밍업 기간만큼 앞에서부터 계산
        calc_start_dt = (pd.to_datetime(last_saved) - BDay(ROLLING_WARMUP_BDAYS)).date().isoformat()
        save_after_dt = pd.to_datetime(last_saved).date()
    else:
        calc_start_dt = min_start_date
        save_after_dt = None

    # 1) DB에서 종목 주가 조회
    px_raw = fetch_price_from_db(
        db_info,
        ticker,
        start_date = calc_start_dt,
        end_date   = today_iso,
    )

    # 2) 데이터 품질 검증
    ok, reason = validate_price_series(px_raw, ticker)
    if not ok:
        return False, reason

    # 양수 가격만 사용
    px_stock = px_raw[px_raw["price"] > 0].copy()
    px_stock = px_stock.rename(columns={"price": "price_stock"})

    # 3) SPY 가격 슬라이싱
    spy2 = spy_price_df.copy()
    spy2["date"] = pd.to_datetime(spy2["date"], errors="coerce")
    spy2 = spy2.dropna(subset=["date", "price_mkt"])
    spy2 = spy2[spy2["date"] >= pd.to_datetime(calc_start_dt)]
    if spy2.empty:
        return False, f"{ticker}: SPY 슬라이스 없음 (from {calc_start_dt})"

    # 4) Beta / Required Return 계산
    feat = build_features(
        price_stock_df = px_stock[["date", "price_stock"]],
        price_mkt_df   = spy2[["date", "price_mkt"]],
        rf_df          = rf_df,
    )
    if feat is None or feat.empty:
        return False, f"{ticker}: feature 계산 결과 없음"

    feat["ticker"] = ticker

    # 5) 저장 컬럼 선택
    if store_mode == "full":
        keep_cols = [
            "price_mkt", "ret_stock", "ret_mkt",
            "beta_252", "beta_750", "beta_1250",
            "rf_1y", "rf_3y", "rf_5y",
            "E_Rm_1y", "E_Rm_3y", "E_Rm_5y",
            "Re_1y", "Re_3y", "Re_5y",
        ]
    else:  # minimal
        keep_cols = [
            "beta_252", "beta_750", "beta_1250",
            "Re_1y", "Re_3y", "Re_5y",
        ]
    keep_cols = [c for c in keep_cols if c in feat.columns]

    # 6) last_saved 이후만 저장 → 중복 upsert 방지
    if save_after_dt is not None:
        feat = feat[pd.to_datetime(feat["date"]).dt.date > save_after_dt]
        if feat.empty:
            return True, f"{ticker}: 이미 최신 (last={save_after_dt})"

    # 7) wide → long melt
    long_df = feat[["date", "ticker"] + keep_cols].melt(
        id_vars    = ["date", "ticker"],
        value_vars = keep_cols,
        var_name   = "indicator",
        value_name = "value",
    )
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["value"])

    if long_df.empty:
        return False, f"{ticker}: 계산된 값 모두 NaN"

    # 8) DB 저장
    n_saved = upsert_long_df(db_info, long_df, drop_null_values=True)
    return True, f"{ticker}: saved {n_saved:,}행 ({store_mode}) from {calc_start_dt}"


print("[OK] update_one_ticker 함수 정의 완료")

[OK] update_one_ticker 함수 정의 완료


## Cell 9 · SPY / RF 공통 데이터 준비

SPY 가격과 RF 는 배치 전 1회만 로드해서 모든 티커에 재사용합니다.  
SPY는 DB 우선 → FDR fallback.

In [9]:
def prepare_spy_and_rf(
    db_info: Dict[str, Any],
    today_iso: str,
    min_start_date: str = MIN_START_DATE,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    SPY 가격 + RF 를 1회 로드 후 반환.
    SPY: DB 우선, 없으면 FDR fallback.

    Returns
    -------
    (spy_price_df, rf_df)
      spy_price_df : columns [date, price_mkt]
      rf_df        : index=date, columns=[rf_1y, rf_3y, rf_5y]
    """
    log("PREP", "SPY 가격 로드 시작")

    # ── SPY: DB 우선 ──────────────────────────────────────────────
    spy_raw = fetch_price_from_db(db_info, MARKET_TICKER, start_date=min_start_date)

    if spy_raw.empty:
        log("PREP", "[WARN] DB에 SPY 없음 → FDR fallback")
        spy_raw = fetch_fdr_price_fallback(MARKET_TICKER, min_start_date, today_iso)

    if spy_raw.empty:
        raise RuntimeError("SPY 가격 확보 실패. DB 및 FDR 상태를 확인하세요.")

    spy_price_df = (
        spy_raw
        .rename(columns={"price": "price_mkt"})
        [["date", "price_mkt"]]
        .copy()
    )
    spy_price_df["date"]      = pd.to_datetime(spy_price_df["date"])
    spy_price_df["price_mkt"] = pd.to_numeric(spy_price_df["price_mkt"], errors="coerce")
    spy_price_df = (
        spy_price_df
        .dropna(subset=["date", "price_mkt"])
        .drop_duplicates("date")
        .sort_values("date")
    )
    log("PREP", f"SPY 가격: {len(spy_price_df)}행  "
               f"({spy_price_df['date'].iloc[0].date()} ~ {spy_price_df['date'].iloc[-1].date()})")

    # ── RF ────────────────────────────────────────────────────────
    start_for_rf = spy_price_df["date"].min().date().isoformat()
    end_for_rf   = spy_price_df["date"].max().date().isoformat()
    log("PREP", f"RF 로드 중: {start_for_rf} ~ {end_for_rf}")
    rf_df = fetch_us_treasury_yields(start_for_rf, end_for_rf)
    if rf_df is None or rf_df.empty:
        log("PREP", "[WARN] RF 로드 실패 → Required Return 이 NaN 이 될 수 있습니다.")

    return spy_price_df, rf_df


print("[OK] prepare_spy_and_rf 함수 정의 완료")

[OK] prepare_spy_and_rf 함수 정의 완료


## Cell 10 · 배치 실행

### 실행 모드
| 변수 | 설명 |
|------|------|
| `RUN_TICKERS` | `None` → 구간/전체 실행 / 리스트 → 해당 티커만 |
| `TICKER_START` | 리스트 슬라이싱 시작 인덱스 (0부터) |
| `TICKER_END` | 슬라이싱 끝 인덱스 (None = 끝까지) |
| `SKIP_DONE` | True → 체크포인트 티커 건너뜀 **(정기 업데이트 시 False 권장)** |
| `RETRY_FAILED` | True → 실패 티커 1회 재시도 |

### ⚠️ 주의
> 정기 업데이트 시 반드시 `SKIP_DONE = False` 로 실행하세요.  
> `SKIP_DONE = True` 는 최초 전체 구축 시에만 사용하세요.

In [10]:
# ══════════════════════════════════════════════════════════════════
#  배치 설정 — 여기를 수정하세요
# ══════════════════════════════════════════════════════════════════

RUN_TICKERS  = None              # None: 구간 실행 / 예: ["AAPL", "MSFT"]

TICKER_START = 0                 # 시작 인덱스
TICKER_END   = None              # 끝 인덱스 (None = 전체)

RUN_STORE_MODE = STORE_MODE      # "minimal" or "full"
SKIP_DONE      = False           # ★ 정기 업데이트: False / 최초 구축: True
RETRY_FAILED   = True            # 실패 티커 재시도

DONE_PATH = DEFAULT_DONE_PATH
FAIL_PATH = DEFAULT_FAIL_PATH

# ══════════════════════════════════════════════════════════════════
#  실행 대상 결정
# ══════════════════════════════════════════════════════════════════
if RUN_TICKERS is not None:
    tickers = RUN_TICKERS
    log("BATCH", f"모드: 특정 티커 지정  {tickers}")
else:
    tickers = DEFAULT_TICKER_LIST[TICKER_START:TICKER_END]
    _s = TICKER_START if TICKER_START is not None else 0
    _e = TICKER_END   if TICKER_END   is not None else len(DEFAULT_TICKER_LIST)
    log("BATCH", f"모드: 구간 실행  index {_s} ~ {_e-1}  ({len(tickers)}개)")

total     = len(tickers)
today_iso = datetime.utcnow().date().isoformat()
done_set  = _load_set(DONE_PATH) if SKIP_DONE else set()
fail_set  = set()

# ── SPY / RF 공통 데이터 준비 (배치 전 1회) ──────────────────────
spy_price_df, rf_df = prepare_spy_and_rf(
    db_info        = db_info,
    today_iso      = today_iso,
    min_start_date = MIN_START_DATE,
)

success, skipped, errored = 0, 0, 0

log("BATCH", "=" * 70)
log("BATCH", f"시작  | {total}개 티커 | store_mode={RUN_STORE_MODE}")
log("BATCH", f"skip_done={SKIP_DONE}  retry_failed={RETRY_FAILED}")
if SKIP_DONE:
    log("BATCH", f"체크포인트 완료: {len(done_set)}개")
log("BATCH", "=" * 70)


def _run_one(ticker: str) -> Tuple[bool, str]:
    """티커 1개 처리. (True, msg) or (False, msg) 반환."""
    try:
        ok, msg = update_one_ticker(
            db_info      = db_info,
            ticker       = ticker,
            spy_price_df = spy_price_df,
            rf_df        = rf_df,
            today_iso    = today_iso,
            min_start_date = MIN_START_DATE,
            store_mode   = RUN_STORE_MODE,
        )
    except Exception as e:
        return False, f"{ticker}: 예외 발생 → {e}"
    finally:
        gc.collect()
    return ok, msg


# ── 메인 루프 ─────────────────────────────────────────────────────
for idx, ticker in enumerate(tickers, 1):
    ticker = str(ticker).strip()
    if not ticker:
        continue

    pct = idx / total * 100
    log("PROGRESS", f"[{idx:>4}/{total}] ({pct:5.1f}%)  >>  {ticker}")

    if SKIP_DONE and ticker in done_set:
        log(ticker, "[SKIP] 이미 완료 (체크포인트)")
        skipped += 1
        continue

    ok, msg = _run_one(ticker)
    if ok:
        log(ticker, f"[OK] {msg}")
        _append_line(DONE_PATH, ticker)
        done_set.add(ticker)
        success += 1
    else:
        log(ticker, f"[FAIL] {msg}")
        _append_line(FAIL_PATH, ticker)
        fail_set.add(ticker)
        errored += 1

# ── 실패 티커 재시도 ──────────────────────────────────────────────
if RETRY_FAILED and fail_set:
    log("RETRY", f"실패 티커 {len(fail_set)}개 재시도")
    still_fail = set()
    for ticker in sorted(fail_set):
        ok, msg = _run_one(ticker)
        if ok:
            log(ticker, f"[RETRY-OK] {msg}")
            _append_line(DONE_PATH, ticker)
            done_set.add(ticker)
            success += 1
            errored -= 1
        else:
            log(ticker, f"[RETRY-FAIL] {msg}")
            still_fail.add(ticker)
    log("RETRY", f"재시도 완료. 최종 실패: {len(still_fail)}개")

# ── 배치 요약 ─────────────────────────────────────────────────────
log("BATCH", "=" * 70)
log("BATCH", f"완료 | 성공={success}  스킵={skipped}  실패={errored}  합계={total}")
log("BATCH", f"체크포인트: {DONE_PATH}")
log("BATCH", "=" * 70)

[16:07:31][BATCH] 모드: 구간 실행  index 0 ~ 1999  (2000개)
[16:07:31][PREP] SPY 가격 로드 시작
[16:07:31][PREP] [WARN] DB에 SPY 없음 → FDR fallback
[16:07:32][PREP] SPY 가격: 2831행  (2014-12-31 ~ 2026-04-06)
[16:07:32][PREP] RF 로드 중: 2014-12-31 ~ 2026-04-06
[16:07:34][RF] Yahoo fallback 로드 성공: 2939행
[16:07:34][BATCH] ======================================================================
[16:07:34][BATCH] 시작  | 2000개 티커 | store_mode=minimal
[16:07:34][BATCH] skip_done=False  retry_failed=True
[16:07:34][BATCH] ======================================================================
[16:07:34][PROGRESS] [   1/2000] (  0.1%)  >>  NVDA
[16:07:49][DB] upsert batch 1: tickers=1, rows=348
[16:07:49][NVDA] [OK] NVDA: saved 348행 (minimal) from 2020-12-04
[16:07:49][PROGRESS] [   2/2000] (  0.1%)  >>  GOOG
[16:08:04][DB] upsert batch 1: tickers=1, rows=342
[16:08:04][GOOG] [OK] GOOG: saved 342행 (minimal) from 2020-12-07
[16:08:04][PROGRESS] [   3/2000] (  0.1%)  >>  AAPL
[16:08:19][DB] upsert batch 1: tickers=1, r

## Cell 11 · DB 조회 유틸 (pivot 조회)

In [11]:
def fetch_required_return_pivot_from_db(
    db_info: Dict[str, Any],
    table_name: str                 = TABLE_RESULT,
    indicators: Optional[List[str]] = None,
    start_date: Optional[str]       = None,
    end_date: Optional[str]         = None,
    tickers: Optional[List[str]]    = None,
) -> pd.DataFrame:
    """
    DB long-format → pivot DataFrame.
    반환 컬럼: date, ticker, [indicator 컬럼들...]

    Notes
    -----
    - pd.read_sql + pymysql DictCursor 버그 방지: cur.execute() 직접 사용
    - CASE WHEN indicator 값은 SQL 리터럴 직접 삽입 (파라미터 순서 꼬임 방지)
    """
    if indicators is None:
        indicators = ["Re_1y", "Re_3y", "Re_5y",
                      "beta_252", "beta_750", "beta_1250"]

    # CASE WHEN 절 생성 (indicator 값을 SQL 리터럴로 직접 삽입)
    case_parts = []
    for ind in indicators:
        ind_esc  = ind.replace("'", "''")
        col_name = ind.replace("'", "''").replace("`", "``")
        case_parts.append(
            f"MAX(CASE WHEN indicator = '{ind_esc}' THEN value END) AS `{col_name}`"
        )
    case_sql = ",\n                ".join(case_parts)

    # WHERE 절 (조건값만 %s 바인딩)
    where_clauses = []
    params        = []

    if start_date:
        where_clauses.append("date >= %s")
        params.append(start_date)
    if end_date:
        where_clauses.append("date <= %s")
        params.append(end_date)
    if tickers:
        placeholders = ", ".join(["%s"] * len(tickers))
        where_clauses.append(f"ticker IN ({placeholders})")
        params.extend(tickers)

    where_sql = ("WHERE " + " AND ".join(where_clauses)) if where_clauses else ""

    query = f"""
        SELECT
            date,
            ticker,
            {case_sql}
        FROM   {table_name}
        {where_sql}
        GROUP  BY date, ticker
        ORDER  BY date, ticker;
    """

    conn = get_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(query, params if params else None)
            rows = cur.fetchall()
            df_pivot = pd.DataFrame(rows)
    finally:
        conn.close()

    if df_pivot.empty:
        return pd.DataFrame(columns=["date", "ticker"] + indicators)

    df_pivot["date"] = pd.to_datetime(df_pivot["date"], errors="coerce")
    df_pivot = df_pivot.dropna(subset=["date"]).reset_index(drop=True)
    return df_pivot


print("[OK] fetch_required_return_pivot_from_db 함수 정의 완료")

[OK] fetch_required_return_pivot_from_db 함수 정의 완료


## Cell 12 · 결과 조회 테스트

In [12]:
# ── 단일 티커 조회 테스트 ─────────────────────────────────────────
TEST_TICKER = "AAPL"   # ← 변경 가능

re_df = fetch_required_return_pivot_from_db(
    db_info    = db_info,
    table_name = TABLE_RESULT,
    indicators = ["Re_1y", "Re_3y", "Re_5y", "beta_252", "beta_750", "beta_1250"],
    start_date = "2020-01-01",
    end_date   = None,
    tickers    = [TEST_TICKER],
)

print(f"[조회] {TEST_TICKER}  총 {len(re_df)}행")
print("\n--- 앞 5행 ---")
display(re_df.head())
print("\n--- 뒤 5행 ---")
display(re_df.tail())

[조회] AAPL  총 1572행

--- 앞 5행 ---


,date,ticker,Re_1y,Re_3y,Re_5y,beta_252,beta_750,beta_1250
0,2020-01-02,AAPL,0.439061,0.197742,0.152094,1.553092,1.369364,1.241175
1,2020-01-03,AAPL,0.433768,0.194447,0.152494,1.447101,1.369372,1.239492
2,2020-01-06,AAPL,0.394956,0.194831,0.150513,1.463324,1.369466,1.243087
3,2020-01-07,AAPL,0.381275,0.194678,0.149214,1.471397,1.369474,1.242719
4,2020-01-08,AAPL,0.375207,0.196122,0.149212,1.471053,1.370458,1.243174



--- 뒤 5행 ---


,date,ticker,Re_1y,Re_3y,Re_5y,beta_252,beta_750,beta_1250
1567,2026-03-30,AAPL,0.165763,0.189276,0.134588,1.279011,1.147052,1.219811
1568,2026-03-31,AAPL,0.227681,0.198795,0.140462,1.272297,1.145143,1.218125
1569,2026-04-01,AAPL,0.228376,0.203839,0.140458,1.270072,1.145312,1.217493
1570,2026-04-02,AAPL,0.225915,0.205145,0.140594,1.269993,1.144909,1.217481
1571,2026-04-06,AAPL,0.224081,0.205507,0.141010,1.271348,1.145217,1.217294


In [13]:
# ── 전체 저장 현황 요약 ───────────────────────────────────────────
conn = get_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT
                indicator,
                COUNT(DISTINCT ticker) AS ticker_cnt,
                COUNT(*)               AS total_rows,
                MIN(date)              AS date_from,
                MAX(date)              AS date_to
            FROM   {TABLE_RESULT}
            GROUP  BY indicator
            ORDER  BY indicator;
        """)
        rows = cur.fetchall()
        summary_df = pd.DataFrame(rows)
finally:
    conn.close()

print(f"[전체 저장 현황]  {TABLE_RESULT}")
display(summary_df)

[전체 저장 현황]  us_required_return_result


,indicator,ticker_cnt,total_rows,date_from,date_to
0,beta_1250,2919,4616778,2014-12-19,2026-04-06
1,beta_252,3599,7883707,2011-01-03,2026-04-06
2,beta_750,3268,6194489,2012-12-26,2026-04-06
3,E_Rm_1y,3,7560,2016-01-04,2026-01-09
4,E_Rm_3y,3,6066,2017-12-22,2026-01-09
5,E_Rm_5y,3,4566,2019-12-19,2026-01-09
6,price_mkt,3,8316,2015-01-02,2026-01-09
7,price_stock,3998,8061714,2014-12-31,2026-04-06
8,ret_mkt,3,8313,2015-01-05,2026-01-09
9,ret_stock,3,8313,2015-01-05,2026-01-09
